# Tutorial 8: Building Reusable Tools

**Programming Design Principles / Maths for IT**

In the first seven tutorials, we learned to write functions. In Skills Demo 1, we used them to build algorithms. Now we are going to think more carefully about how to *design* functions -- not just to solve a specific problem, but to create tools we can reuse and combine.

This tutorial is about the craft of writing good functions. It connects to a professional practice called *modular design*, where complex programs are built from small, independent, well-tested pieces.

## What Makes a Good Function?

A good function does one thing, does it well, and communicates clearly what it does. Let's look at an example and think about what makes it work:

In [1]:
def mean(numbers):
    """Compute the arithmetic mean of a list of numbers.
    
    Parameters:
        numbers: a list of numeric values (must not be empty)
    
    Returns:
        the arithmetic mean as a float
    """
    total = 0
    for value in numbers:
        total = total + value
    return total / len(numbers)

# Test it
print(mean([10, 20, 30]))        # should be 20.0
print(mean([1, 2, 3, 4, 5]))     # should be 3.0

20.0
3.0


That triple-quoted string at the top of the function is called a *docstring*. It describes what the function does, what it expects as input, and what it returns. This is not just decoration -- it is how professional programmers communicate the *contract* of a function. Anyone who wants to use `mean()` can read the docstring and know exactly what to pass in and what they will get back.

From now on, every function we write should have a docstring. It does not need to be elaborate -- a single clear sentence is often enough -- but it should be there.

## Functions Calling Functions

The real power of modular design appears when functions use other functions as building blocks. Here is a function that computes the standard deviation -- and notice how it uses `mean()`:

In [2]:
def std_dev(numbers):
    """Compute the population standard deviation of a list of numbers."""
    avg = mean(numbers)
    squared_diffs = []
    for value in numbers:
        diff = value - avg
        squared_diffs.append(diff ** 2)
    return mean(squared_diffs) ** 0.5

print(std_dev([10, 20, 30]))

8.16496580927726


We did not rewrite the averaging logic inside `std_dev`. We called `mean()` twice -- once for the original average, and once to average the squared differences. This is the essence of modular design: build small tools and combine them.

If we later discover a bug in `mean()`, we fix it once and `std_dev()` automatically benefits. If we want to use `mean()` somewhere else, it is already available. Each function is an independent, tested unit.

### Your turn

Write a function `data_range(numbers)` that returns the difference between the largest and smallest values in a list. Then write a function `describe(numbers)` that calls `mean()`, `std_dev()`, and `data_range()` to print a summary of the data.

Include docstrings for both functions.

In [3]:
# Pseudocode for data_range:
# FIND the largest value in the list
# FIND the smallest value in the list
# RETURN the largest minus the smallest

# Your data_range function
def data_range(numbers):
    """Return the difference between the largest and smallest values in numbers."""
    return max(numbers) - min(numbers)


In [4]:
# Your describe function
def describe(numbers):
    """Print a summary of numbers: mean, standard deviation, and range."""
    print("Mean:", mean(numbers))
    print("Standard deviation:", std_dev(numbers))
    print("Range:", data_range(numbers))


In [5]:
# Test describe with some data
test_data = [42, 38, 35, 47, 29, 41, 44, 33, 39, 48]

describe(test_data)


Mean: 39.6
Standard deviation: 5.76541412215983
Range: 19


## Handling Edge Cases

What happens if someone calls `mean([])` -- with an empty list? Division by zero. A good function anticipates this:

In [6]:
def safe_mean(numbers):
    """Compute the arithmetic mean, returning None for empty lists."""
    if len(numbers) == 0:
        return None
    total = 0
    for value in numbers:
        total = total + value
    return total / len(numbers)

print(safe_mean([1, 2, 3]))
print(safe_mean([]))

2.0
None


Returning `None` for invalid input is one common approach. Another is to print a clear error message. The important thing is that the function does not crash silently or return a misleading result.

### Your turn

Go back to your `data_range` function. What happens with an empty list? A list with one element? Update it to handle these cases gracefully.

In [7]:
# Updated data_range with edge case handling
def data_range(numbers):
    """Return the difference between the largest and smallest values in numbers.

    Returns None for an empty list. A single-element list has a range of 0.
    """
    if len(numbers) == 0:
        return None
    return max(numbers) - min(numbers)

print(data_range([1, 5, 3]))
print(data_range([7]))
print(data_range([]))


4
0
None


## Variable Scope Revisited

Now that we are writing functions that call other functions, let's make sure we understand scope. Each function has its own workspace. Variables created inside a function vanish when the function finishes:

In [8]:
def add_tax(price, rate):
    tax = price * rate
    total = price + tax
    return total

result = add_tax(100, 0.23)
print("Total:", result)

# These would cause errors if uncommented:
# print(tax)      # does not exist here
# print(total)    # does not exist here either

Total: 123.0


This is a feature: it means you can use the name `total` inside multiple different functions without them interfering with each other. Each function's `total` is its own separate variable.

The way functions communicate is through *parameters* (values passed in) and *return values* (values sent back). This is cleaner and more reliable than sharing global variables.

### Your turn

Write two functions that each use a variable called `count` internally but for different purposes. Verify that they do not interfere with each other.


In [9]:
# Two functions that both use 'count' internally
def count_vowels(word):
    """Count the number of vowels in a word."""
    count = 0
    for letter in word.lower():
        if letter in "aeiou":
            count = count + 1
    return count

def count_evens(numbers):
    """Count how many numbers in a list are even."""
    count = 0
    for value in numbers:
        if value % 2 == 0:
            count = count + 1
    return count


In [10]:
# Demonstrate they work independently
print(count_vowels("programming"))
print(count_evens([1, 2, 3, 4, 5, 6]))


3
3


## Testing as a Habit

So far we have been testing informally: run the function, check the output by eye. Let's make this more systematic. A good test checks that a function produces the expected output for a known input:

In [11]:
def test_mean():
    """Test the mean function with known cases."""
    # Basic case
    result = mean([10, 20, 30])
    expected = 20.0
    if result == expected:
        print("PASS: mean([10, 20, 30]) = " + str(result))
    else:
        print("FAIL: mean([10, 20, 30]) expected " + str(expected) + " got " + str(result))
    
    # Single element
    result = mean([42])
    expected = 42.0
    if result == expected:
        print("PASS: mean([42]) = " + str(result))
    else:
        print("FAIL: mean([42]) expected " + str(expected) + " got " + str(result))
    
    # Negative numbers
    result = mean([-10, 10])
    expected = 0.0
    if result == expected:
        print("PASS: mean([-10, 10]) = " + str(result))
    else:
        print("FAIL: mean([-10, 10]) expected " + str(expected) + " got " + str(result))

test_mean()

PASS: mean([10, 20, 30]) = 20.0
PASS: mean([42]) = 42.0
PASS: mean([-10, 10]) = 0.0


Writing tests like this before or alongside your functions is one of the most valuable habits you can develop. It forces you to think clearly about what the function should do, and it gives you confidence that the function actually does it.

### Your turn

Write a test function for your `data_range` function. Include at least four test cases: a normal list, a list where all elements are the same, a list with negative numbers, and a single-element list.

In [12]:
# Your test_data_range function
def test_data_range():
    """Test the data_range function with known cases."""
    result = data_range([10, 20, 30])
    expected = 20
    if result == expected:
        print("PASS: data_range([10, 20, 30]) = " + str(result))
    else:
        print("FAIL: data_range([10, 20, 30]) expected " + str(expected) + " got " + str(result))

    result = data_range([5, 5, 5])
    expected = 0
    if result == expected:
        print("PASS: data_range([5, 5, 5]) = " + str(result))
    else:
        print("FAIL: data_range([5, 5, 5]) expected " + str(expected) + " got " + str(result))

    result = data_range([-10, 5, -3])
    expected = 15
    if result == expected:
        print("PASS: data_range([-10, 5, -3]) = " + str(result))
    else:
        print("FAIL: data_range([-10, 5, -3]) expected " + str(expected) + " got " + str(result))

    result = data_range([42])
    expected = 0
    if result == expected:
        print("PASS: data_range([42]) = " + str(result))
    else:
        print("FAIL: data_range([42]) expected " + str(expected) + " got " + str(result))

test_data_range()


PASS: data_range([10, 20, 30]) = 20
PASS: data_range([5, 5, 5]) = 0
PASS: data_range([-10, 5, -3]) = 15
PASS: data_range([42]) = 0


## Reflection

Today was less about new Python syntax and more about *how to think* when writing functions: single responsibility, docstrings, edge cases, scope discipline, and systematic testing. These are the practices that separate code that works once from code that can be relied upon.

In the next tutorials, we will use these practices to build tools for counting, probability, and statistics -- each one a well-documented, well-tested function that becomes part of our growing toolkit.

What feels different about thinking of functions as *tools* versus thinking of them as *solutions to homework problems*?

